# Phase 0: Data Ingestion, Cleaning & Synchronization

This notebook implements the Quality Control protocol for the QMUL sensor network. It removes detectable sensor faults, filters out high-frequency noise, performs light-touch temporal imputation, and synchronizes data for high-resolution analysis and calibration.

## Objectives
1. **Quality Control**: Applies strict environmental bounds, internal consistency checks, consecutive stuck reading detection (>=10 samples), and double-pass Hampel filtering for spike removal (Low-cost sensors only).
2. **Resampling & Imputation**: Standardize low-cost data to a 1-minute grid and a 15-minute grid. Apply linear interpolation **only** to the 1-minute grid (strictly to gaps <= 1 minute).
3. **Verification**: Calculate empirical gap lengths to justify imputation boundaries and summarize dropped data.
4. **Synchronization**: Create a 1-minute master file for low-cost nodes, and a 15-minute combined master file linking low-cost networks with reference station data.

In [1]:
import pandas as pd
import numpy as np
import os
import glob
from tqdm.auto import tqdm
from IPython.display import display

# Configuration
NODE_DIR = "../data/01_raw/node_data"
REF_DIR = "../data/01_raw/reference_data"
OUTPUT_DIR = "../data/02_interim"

MAX_IMPUTE_MINUTES = 1

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. QC Protocol
This function applies the QC protocol to raw low-cost sensor data. 

The protocol includes:
1. **Physical/Environmental Bounds:** Hard limits to remove mathematically impossible or physically unreasonable values (e.g., negative PM, Temp outside -10 to 70°C).
2. **Internal Consistency:** Ensuring PM10 is strictly $\ge$ PM2.5.
3. **Stuck Reading Detection:** Identifying and masking instances where the sensor reports $\ge 10$ consecutive identical values, indicating a hardware freeze.
4. **Spike Detection (Hampel Filters):** 
   - **Pass 1:** A Hampel filter (window $h=20$, threshold $k=4.0$ MAD) applied to all meteorological and PM columns to remove high-frequency noise and electronic artifacts.
   - **Pass 2:** A secondary Hampel filter applied strictly to PM columns to catch residual local plume artifacts.

High-cost reference data bypasses these strict filters (only negative values are removed).

In [2]:
def hampel_filter(s, window_size=21, k=4.0):
    rolling_median = s.rolling(window=window_size, center=True, min_periods=1).median()
    mad = (s - rolling_median).abs().rolling(window=window_size, center=True, min_periods=1).median()
    outliers = (s - rolling_median).abs() > (k * 1.4826 * mad)
    s_out = s.copy()
    s_out[outliers] = np.nan
    return s_out, outliers.sum()

def remove_stuck(s, min_consecutive=10):
    s_valid = s.dropna()
    if s_valid.empty: return s, 0
    blocks = (s_valid != s_valid.shift()).cumsum()
    counts = s_valid.groupby(blocks).transform('count')
    stuck_mask = counts >= min_consecutive
    
    full_stuck_mask = pd.Series(False, index=s.index)
    full_stuck_mask.loc[s_valid.index] = stuck_mask
    s_out = s.copy()
    s_out[full_stuck_mask] = np.nan
    return s_out, full_stuck_mask.sum()

def apply_mvc_raw(df, node_id):
    """
    Applies the QC protocol.
    Includes hardware bounds, internal consistency, stuck reading checks, and Hampel filtering.
    """
    # Step 0: Metadata cleanup and indexing
    cols_to_drop = ['latitude', 'longitude', 'site', 'subsite', 'exposure', 'environment', 'node_id']
    df = df.drop(columns=cols_to_drop, errors='ignore')
    df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'])
    df = df.set_index('timestamp_utc').sort_index()
    
    is_ref = 'TH' in node_id
    report = {'node': node_id, 'Total Rows': len(df)}
    
    # --- Step 1: Physical / Environmental Bounds ---
    pre = df.isna().sum().sum()
    if not is_ref:
        if 'pm2_5_low_sps30_ug_m3' in df.columns:
            df.loc[(df['pm2_5_low_sps30_ug_m3'] < 0) | (df['pm2_5_low_sps30_ug_m3'] > 1000), 'pm2_5_low_sps30_ug_m3'] = np.nan
        if 'pm10_low_sps30_ug_m3' in df.columns:
            df.loc[(df['pm10_low_sps30_ug_m3'] < 0) | (df['pm10_low_sps30_ug_m3'] > 1000), 'pm10_low_sps30_ug_m3'] = np.nan
        if 'temperature_low_sht45_celsius' in df.columns:
            df.loc[(df['temperature_low_sht45_celsius'] < -10) | (df['temperature_low_sht45_celsius'] > 70), 'temperature_low_sht45_celsius'] = np.nan
        if 'humidity_low_sht45_percentage' in df.columns:
            df.loc[(df['humidity_low_sht45_percentage'] < 0) | (df['humidity_low_sht45_percentage'] > 100), 'humidity_low_sht45_percentage'] = np.nan
    else:
        if 'pm2_5_high_met_one_ug_m3' in df.columns:
            df.loc[(df['pm2_5_high_met_one_ug_m3'] < 0) | (df['pm2_5_high_met_one_ug_m3'] > 1000), 'pm2_5_high_met_one_ug_m3'] = np.nan
    report['Range Drops (Cells)'] = df.isna().sum().sum() - pre

    if is_ref:
        return df, report

    # --- Step 2: Consistency: PM10 >= PM2.5 ---
    pre = df.isna().sum().sum()
    if 'pm10_low_sps30_ug_m3' in df.columns and 'pm2_5_low_sps30_ug_m3' in df.columns:
        mask = df['pm10_low_sps30_ug_m3'] < df['pm2_5_low_sps30_ug_m3']
        df.loc[mask, ['pm2_5_low_sps30_ug_m3', 'pm10_low_sps30_ug_m3']] = np.nan
    report['Consistency Drops (Cells)'] = df.isna().sum().sum() - pre

    # --- Step 3: Stuck Readings (>= 10 consecutive identical) ---
    pre = df.isna().sum().sum()
    for col in ['pm2_5_low_sps30_ug_m3', 'pm10_low_sps30_ug_m3', 'temperature_low_sht45_celsius', 'humidity_low_sht45_percentage']:
        if col in df.columns:
            df[col], _ = remove_stuck(df[col], 10)
    report['Persistence Drops (Cells)'] = df.isna().sum().sum() - pre

    # --- Step 4: Spikes (Hampel Pass 1: All cols) ---
    pre = df.isna().sum().sum()
    for col in ['pm2_5_low_sps30_ug_m3', 'pm10_low_sps30_ug_m3', 'temperature_low_sht45_celsius', 'humidity_low_sht45_percentage']:
        if col in df.columns:
            df[col], _ = hampel_filter(df[col], 21, 4.0)
    report['Hampel 1 Drops'] = df.isna().sum().sum() - pre

    # --- Step 5: Spikes (Hampel Pass 2: PM Only) ---
    pre = df.isna().sum().sum()
    for col in ['pm2_5_low_sps30_ug_m3', 'pm10_low_sps30_ug_m3']:
        if col in df.columns:
            df[col], _ = hampel_filter(df[col], 21, 4.0)
    report['Hampel 2 Drops'] = df.isna().sum().sum() - pre

    return df, report

## 2. Network Processing Loop & Gap Analysis
We clean each low-cost file, calculate empirical gap lengths to justify our imputation bounds, apply the linear imputation strictly to the 1-minute grid, and build the 15-minute calibration sheet without imputation.

In [3]:
node_files = sorted(glob.glob(os.path.join(NODE_DIR, "node_*_filtered.csv")))
ref_files = sorted(glob.glob(os.path.join(REF_DIR, "node_*_filtered.csv")))

nodes_1m, nodes_15m = [], []
cleaning_log, gap_log = [], []

# 2.1 Process Low-Cost Nodes
for f in tqdm(node_files, desc="Low-Cost Nodes"):
    node_id = os.path.basename(f).replace("_filtered.csv", "")
    df_raw = pd.read_csv(f)
    
    # Apply QC protocol (Steps 1 to 3)
    clean_df, report = apply_mvc_raw(df_raw, node_id)
    cleaning_log.append(report)
    
    # Rename columns to standard identifiers
    rename = {
        'pm2_5_low_sps30_ug_m3': f'{node_id}_pm25', 
        'pm10_low_sps30_ug_m3': f'{node_id}_pm10',
        'temperature_low_sht45_celsius': f'{node_id}_temp', 
        'humidity_low_sht45_percentage': f'{node_id}_humi'
    }
    clean_df = clean_df.rename(columns=rename)
    
    # Step 4: Resampling (NO IMPUTATION YET)
    df_1m = clean_df.resample('1min').mean().round(3)
    df_15m = clean_df.resample('15min').mean().round(3)
    
    # --- Gap Analysis (Calculated before imputation) ---
    col = f'{node_id}_pm25'
    if col in df_1m.columns:
        # Find gaps for 1min
        is_na_1m = df_1m[col].isna()
        gaps_1m = is_na_1m.groupby((~is_na_1m).cumsum()).sum()
        gaps_1m = gaps_1m[gaps_1m > 0]
        
        # Find gaps for 15min
        is_na_15m = df_15m[col].isna()
        gaps_15m = is_na_15m.groupby((~is_na_15m).cumsum()).sum()
        gaps_15m = gaps_15m[gaps_15m > 0]
        
        gap_log.append({
            'node': node_id,
            'Total Gap Instances': len(gaps_1m),
            '1min_median_gap': gaps_1m.median() if not gaps_1m.empty else 0,
            '1min_max_gap': gaps_1m.max() if not gaps_1m.empty else 0,
            '15min_median_gap': gaps_15m.median() if not gaps_15m.empty else 0,
            '15min_max_gap': gaps_15m.max() if not gaps_15m.empty else 0
        })
    
    # Step 5: Temporal Imputation (APPLIED ONLY TO 1-MINUTE GRID)
    df_1m_imputed = df_1m.interpolate(method='linear', limit=MAX_IMPUTE_MINUTES)
    
    nodes_1m.append(df_1m_imputed)
    nodes_15m.append(df_15m)  # Keep 15m unimputed

# 2.2 Process Reference Stations
refs_15m = []
for f in tqdm(ref_files, desc="Reference Stations"):
    node_id = os.path.basename(f).replace("_filtered.csv", "")
    df_raw = pd.read_csv(f)
    
    # Apply QC protocol (Step 1 will remove negative PM2.5 values)
    clean_df, report = apply_mvc_raw(df_raw, node_id)
    cleaning_log.append(report)
    
    # Rename column to standard identifier
    clean_id = node_id.replace('node_', '').lower()
    rename = {'pm2_5_high_met_one_ug_m3': f"{clean_id}_pm25"}
    clean_df = clean_df.rename(columns=rename)
    
    # Resample and round to match Ref raw precision (1 decimal)
    df_res = clean_df.select_dtypes(include=[np.number]).resample('15min').mean().round(1)
    refs_15m.append(df_res)

# 2.3 Master Synchronization (Outer Joins)
master_1min = pd.concat(nodes_1m, axis=1, join='outer').dropna(how='all')
master_15min = pd.concat(nodes_15m + refs_15m, axis=1, join='outer').dropna(how='all')

# Export
master_1min.to_csv(os.path.join(OUTPUT_DIR, "master_low_cost_1min.csv"))
master_15min.to_csv(os.path.join(OUTPUT_DIR, "master_combined_15min.csv"))


Low-Cost Nodes:   0%|          | 0/7 [00:00<?, ?it/s]

Reference Stations:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
print("\n--- Phase 0 Complete ---")

print("\n1. Cleaning Audit Summary (Dropped Values per Step):")
df_cleaning_log = pd.DataFrame(cleaning_log).set_index('node')
display(df_cleaning_log)

print("\n2. Gap Analysis Summary (Calculated Prior to Imputation):")
print("Note: Lengths are in units of the resampled grid (e.g., 1 unit = 1 minute or 15 minutes).")
df_gap_log = pd.DataFrame(gap_log).set_index('node')
display(df_gap_log)


--- Phase 0 Complete ---

1. Cleaning Audit Summary (Dropped Values per Step):


,Total Rows,Range Drops (Cells),Consistency Drops (Cells),Persistence Drops (Cells),Hampel 1 Drops,Hampel 2 Drops
node,,,,,,
node_1,856093,0,722.0,0.0,45474.0,2307.0
node_2,1063563,0,0.0,150.0,74463.0,1079.0
node_3,713300,9,0.0,15.0,69747.0,4582.0
node_4,700257,0,0.0,0.0,79509.0,763.0
node_5,660664,0,214.0,36.0,36988.0,1408.0
node_6,1020218,0,0.0,112.0,60381.0,2396.0
node_7,977201,0,0.0,147.0,157170.0,1725.0
node_TH2_MER,40277,812,NaN,NaN,NaN,NaN
node_TH7_KEMP,40315,80,NaN,NaN,NaN,NaN



2. Gap Analysis Summary (Calculated Prior to Imputation):
Note: Lengths are in units of the resampled grid (e.g., 1 unit = 1 minute or 15 minutes).


,Total Gap Instances,1min_median_gap,1min_max_gap,15min_median_gap,15min_max_gap
node,,,,,
node_1,16252,1.0,6812,1.5,454
node_2,1028,1.0,13825,13.0,920
node_3,10622,1.0,47449,2.0,3162
node_4,2313,1.0,70300,2.5,4686
node_5,54482,1.0,35203,2.0,2346
node_6,4400,1.0,6412,147.5,426
node_7,5449,1.0,4429,7.0,294
